In [1]:
import os
import sys
import json

try:
    import google.colab
    from google.colab import drive
    from google.colab import userdata
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")
    drive.mount('/content/drive', force_remount=True)
    
    !pip install -q tqdm requests openpyxl pandas scikit-learn
    
    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    try:
        API_KEY = userdata.get('YANDEX_API_KEY')
        FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')
    except userdata.SecretNotFoundError:
        raise ValueError("Please set YANDEX_API_KEY and YANDEX_FOLDER_ID in Colab Secrets!")

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    secrets_path = os.path.join(ROOT_DIR, "yandex_secrets.json")
    if os.path.exists(secrets_path):
        with open(secrets_path, "r", encoding="utf-8") as f:
            secrets = json.load(f)
            API_KEY = secrets.get("YANDEX_API_KEY")
            FOLDER_ID = secrets.get("YANDEX_FOLDER_ID")
            if not API_KEY or not FOLDER_ID:
                raise ValueError("YANDEX_API_KEY or YANDEX_FOLDER_ID is missing in yandex_secrets.json")
    else:
        raise FileNotFoundError(f"Please create {secrets_path} with your API keys!")

DATA_DIR = os.path.join(ROOT_DIR, "data")
YANDEX_DB_PATH = os.path.join(ROOT_DIR, "yandex_results_db.json")
YANDEX_EXCEL_PATH = os.path.join(ROOT_DIR, "yandex_results_table.xlsx")

os.makedirs(DATA_DIR, exist_ok=True)

We are running locally!


In [2]:
import time
import requests
import datetime
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

try:
    from extract_labels_standardized import extract_labels_standardized, ID_TO_LABEL
except ImportError:
    raise ImportError("Error: Could not import extract_labels_standardized. Make sure the file exists in your ROOT_DIR.")

EXPERIMENT_NAME = "YandexGPT_Pro_FewShot_v1"

TEST_DATASETS = {
    "General_Test": os.path.join(DATA_DIR, "bench_test_all.json"),
    "LORuGEC_Test": os.path.join(DATA_DIR, "bench_test_lorugec.json")
}

In [3]:
def prompt_yandex_to_restore(unpunctuated_text):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}",
        "x-folder-id": FOLDER_ID
    }
    
    system_prompt = (
        "Ты — профессиональный редактор русского языка. "
        "Твоя задача — расставить знаки препинания в тексте, где они полностью удалены. "
        "Строго соблюдай два правила:\n"
        "1. Ни в коем случае не изменяй, не удаляй и не добавляй новые слова. Только вставляй пунктуацию.\n"
        "2. Выведи только восстановленный текст, без каких-либо комментариев и кавычек."
    )
    
    body = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.0,
            "maxTokens": "1000"
        },
        "messages":[
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": "Мама мыла раму а папа читал газету"},
            {"role": "assistant", "text": "Мама мыла раму, а папа читал газету."},
            {"role": "user", "text": "он сказал что придет завтра"},
            {"role": "assistant", "text": "Он сказал, что придет завтра."},
            {"role": "user", "text": unpunctuated_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=body)
        if response.status_code == 200:
            result = response.json()
            return result['result']['alternatives'][0]['message']['text']
        else:
            print(f"API Error {response.status_code}: {response.text}")
            return unpunctuated_text # Fallback if error
    except Exception as e:
        print(f"Request failed: {e}")
        return unpunctuated_text


In [4]:
def evaluate_yandex_on_dataset(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
        
    all_true_tags = []
    all_pred_tags =[]
    
    # # Logging to txt
    # log_file_path = os.path.join(ROOT_DIR, f"{EXPERIMENT_NAME}_log.txt")
    # log_f = open(log_file_path, 'a', encoding='utf-8')
    # log_f.write(f"\n{'='*50}\nEVALUATING DATASET: {os.path.basename(json_path)}\n{'='*50}\n")
    for idx, item in enumerate(tqdm(dataset, desc=f"Evaluating {os.path.basename(json_path)}")):
        true_tokens = item["tokens"]
        true_labels = item["ner_tags"] 
        
        unpunctuated_text = " ".join(true_tokens)
        llm_output_text = prompt_yandex_to_restore(unpunctuated_text)
        
        if idx < 3:
            print(f"\n[Preview {idx+1} from {os.path.basename(json_path)}]")
            print(f"--- sent text: {unpunctuated_text}")
            print(f"--- received answer: {llm_output_text}")
            
        # # logging to txt
        # log_entry = (
        #     f"--- sent text: {unpunctuated_text}\n"
        #     f"--- received answer: {llm_output_text}\n"
        #     f"--------------------------------------------------\n"
        # )
        # log_f.write(log_entry) 
        
        llm_pairs = extract_labels_standardized(llm_output_text)
        llm_pred_labels =[p['label'] for p in llm_pairs]
        
        aligned_pred_labels =[]
        for i in range(len(true_labels)):
            if i < len(llm_pred_labels):
                aligned_pred_labels.append(llm_pred_labels[i])
            else:
                aligned_pred_labels.append(0) 
                
        for t, p in zip(true_labels, aligned_pred_labels):
            if t != -100: 
                all_true_tags.append(ID_TO_LABEL.get(t, "O"))
                all_pred_tags.append(ID_TO_LABEL.get(p, "O"))
                
        time.sleep(0.6) # yandex rate limits
        
    # # logging to txt
    # log_f.close()

    return classification_report(all_true_tags, all_pred_tags, output_dict=True, zero_division=0)


current_experiment_results = {"timestamp": str(datetime.datetime.now())}

for test_name, file_path in TEST_DATASETS.items():
    print(f"\nStarting {test_name} with {EXPERIMENT_NAME}...")
    
    report = evaluate_yandex_on_dataset(file_path)
    
    current_experiment_results[test_name] = report
    print(f"\nFinished {test_name}. F1-Score: {report['weighted avg']['f1-score']:.4f}")



Starting General_Test with YandexGPT_Pro_FewShot_v1...


Evaluating bench_test_all.json:   0%|          | 0/569 [00:00<?, ?it/s]


[Preview 1 from bench_test_all.json]
--- sent text: В советский период времени число ИТ специалистов в Армении составляло около десяти тысяч Доставшийся в наследство от советского периода времени промышленный и интеллектуальный потенциал оказался благом и горем страны С одной стороны квалифицированные кадры и развитая производственная инфраструктура резко отличали Армению от других регионов СССР где доминировали добывающие отрасли а экономика строилась на поставке сырьевых ресурсов С другой оставшись без сырья для промышленных предприятий энергоресурсов и рынков сбыта продукции конкурентоспособной только на постсоветском пространстве Армения быстро потеряла темпы своего экономического развития Помощь этой стране обычно поступает извне Запад во главе с США регулярно выделяет значительные суммы для улучшения экономического положения в стране Не последнее место в выбивании кредитов для Армении играет армянская диаспора в США и других развитых странах Вехи культурного развития Армении Суд

Evaluating bench_test_lorugec.json:   0%|          | 0/603 [00:00<?, ?it/s]


[Preview 1 from bench_test_lorugec.json]
--- sent text: Вася попробовал и так и эдак но у него все равно ничего не вышло
--- received answer: Вася попробовал и так и эдак, но у него всё равно ничего не вышло.

[Preview 2 from bench_test_lorugec.json]
--- sent text: Кругом лес ни конца ни края
--- received answer: Кругом лес — ни конца ни края.

[Preview 3 from bench_test_lorugec.json]
--- sent text: От него целый год не было ни слуху ни духу
--- received answer: От него целый год не было ни слуху ни духу.

Finished LORuGEC_Test. F1-Score: 0.9591


In [5]:
if os.path.exists(YANDEX_DB_PATH):
    with open(YANDEX_DB_PATH, "r", encoding="utf-8") as f:
        yandex_db = json.load(f)
else:
    yandex_db = {}

yandex_db[EXPERIMENT_NAME] = current_experiment_results

with open(YANDEX_DB_PATH, "w", encoding="utf-8") as f:
    json.dump(yandex_db, f, indent=4, ensure_ascii=False)

print(f"\nResults saved to JSON: {YANDEX_DB_PATH}")

summary_records =[]
detailed_records =[]

for model_name, data in yandex_db.items():
    # Summary Table
    sum_row = {"Experiment": model_name}
    for test_name in TEST_DATASETS.keys():
        if test_name in data and "weighted avg" in data[test_name]:
            metrics = data[test_name]["weighted avg"]
            sum_row[(test_name, "F1")] = round(metrics["f1-score"] * 100, 2)
            sum_row[(test_name, "Precision")] = round(metrics["precision"] * 100, 2)
            sum_row[(test_name, "Recall")] = round(metrics["recall"] * 100, 2)
    summary_records.append(sum_row)
    
    # Detailed Table
    labels = set()
    for test_name in TEST_DATASETS.keys():
        if test_name in data:
            labels.update([k for k in data[test_name].keys() if k not in["accuracy", "macro avg", "weighted avg", "timestamp"]])
    
    for label in sorted(labels):
        det_row = {"Experiment": model_name, "Punctuation": label}
        for test_name in TEST_DATASETS.keys():
            if test_name in data and label in data[test_name]:
                metrics = data[test_name][label]
                det_row[(test_name, "F1")] = round(metrics["f1-score"] * 100, 2)
                det_row[(test_name, "Precision")] = round(metrics["precision"] * 100, 2)
                det_row[(test_name, "Recall")] = round(metrics["recall"] * 100, 2)
                det_row[(test_name, "Support")] = metrics["support"]
        detailed_records.append(det_row)

df_summary = pd.DataFrame(summary_records)
df_detailed = pd.DataFrame(detailed_records)

if not df_summary.empty:
    df_summary.set_index("Experiment", inplace=True)
    df_summary.columns = pd.MultiIndex.from_tuples(df_summary.columns)

if not df_detailed.empty:
    df_detailed.set_index(["Experiment", "Punctuation"], inplace=True)
    df_detailed.columns = pd.MultiIndex.from_tuples(df_detailed.columns)

with pd.ExcelWriter(YANDEX_EXCEL_PATH, engine='openpyxl') as writer:
    if not df_summary.empty:
        df_summary.to_excel(writer, sheet_name="Summary")
    if not df_detailed.empty:
        df_detailed.to_excel(writer, sheet_name="Per-Class Details")

print(f"Exported Yandex experiments to: {YANDEX_EXCEL_PATH}")



Results saved to JSON: /home/temari/god please no diploma/restore_punct/yandex_results_db.json
Exported Yandex experiments to: /home/temari/god please no diploma/restore_punct/yandex_results_table.xlsx
